# 05 — ZIT (Gu 2024 구조 기반 적응 구현) 단일 모델

ZI-Tweedie + LightGBM EM(`ZITboostEQLRegressor`, arXiv:2405.14990)을 돌려 RMSE만 본다. 후처리 없음.

> **충실도 라벨**: "Gu 2024 논문 100% 충실"이 아니라 **"논문 구조를 따른 프로젝트용 근사/적응 구현"**이다. 아래 ✅/⚠️ 참고.

- **논문 구조와 일치(✅)**
  - F_π(cross_entropy) · F_μ(tweedie) · F_φ(EQL/saddlepoint = Tweedie unit deviance) 3-함수 Generalized EM (Algorithm 1)
  - 초기화 스칼라부: μ₀=mean(y>0), φ₀=Σ D_ζ(y;μ₀)/n_pos — 논문 스칼라 식과 일치
  - **ζ(Tweedie power)는 고정이 아니라 profile likelihood로 추정(Algorithm 2)**: 후보 ζ로 EM→train 로그우도 최대 ζ* 선택
  - 최종 예측 E[Y] = (1-π)·μ
- **논문과 다른/근사(⚠️)**
  - 초기화 GBT부: 논문은 스칼라 뒤 *초기 GBT F̂μ^(0) 등을 따로 적합*하나, 여기선 그 단계를 첫 EM 반복에 접어넣은 **근사**
  - 로그우도/φ: exact Tweedie가 아니라 **EQL/saddlepoint 근사**(논문도 EQL을 쓰지만 '근사'임을 명시)
  - ζ 그리드 간격(0.1)은 논문 verbatim이 아니라 임의
  - exposure w_i=1(노출량 없음), unit target→die broadcast→die 평균 집계: die/unit 구조에 맞춘 프로젝트 적응(논문 관측단위 likelihood와 1:1 아님)
  - 수식은 arXiv HTML 전사 기준이며 PDF 원문 한 줄 대조는 아직 아님
- **파라미터**: ZIT 내부 μ/π/φ LightGBM 3개는 **라이브러리 기본값**(HP는 논문이 지정하지 않는 축). ζ만 추정, n_em_iters=10 고정.
- **전처리**: 트리 공통 `PP_FIXED` + `CLIP_Y_EXTREME` + meta features — `04_default_compare`와 동일
- **후처리 없음**: die 예측 (1-π)μ → unit `mean` 집계만 (τ_π·position·zero_clip·postprocess 전부 미적용)
- **출력**: `4_output/0_baseline/default_compare/results_zit.csv` (1행)

## 1. 환경 설정 + import

In [1]:
import os, sys

# Google Drive 파일 ID (Colab 자동 다운로드용, 로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'   # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'   # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'   # preprocessing.zip
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (ZITboostEQLRegressor 추가본 재업로드 필요)

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/zit.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리(2_preprocessing) + 모델링(3_modeling)을 패키지 접두사로 import 하도록 경로 추가
for _d in (os.path.join(PROJECT_ROOT, '2_preprocessing'), os.path.join(PROJECT_ROOT, '3_modeling')):
    if _d not in sys.path:
        sys.path.insert(0, _d)

from modules import preprocess                  # 고정 전처리 래퍼 (cleaning + spatial impute + outlier)
from modules.zit import ZITboostEQLRegressor    # Gu 2024(arXiv:2405.14990) 충실 ZI-Tweedie (+ score_loglik)
from meta_features import add_meta_features      # die_xy / position 메타피처

from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print('model: ZITboostEQLRegressor (EM=Algorithm1, ζ profile=Algorithm2)')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
model: ZITboostEQLRegressor (EM=Algorithm1, ζ profile=Algorithm2)


## 2. 설정 — ZIT / 트리 고정 전처리 / LightGBM 라이브러리 기본값

`PP_FIXED`·`CLIP_Y_EXTREME`은 `04_default_compare`와 동일. ZIT 내부 μ/π/φ LightGBM 3개는 **라이브러리 기본값**.
ζ(Tweedie power)는 '기본값 고정'이 아니라 **`ZETA_GRID` 후보를 profile likelihood로 비교해 추정**(Algorithm 2).

In [2]:
N_FOLDS = 5
N_JOBS  = -1   # ZIT 내부 LightGBM 스레드 수 (단독 실행이면 14). strategy_common §8

CLIP_Y_EXTREME = True   # train y의 max(=1.0, 1건)를 두 번째로 큰 값으로 clip (학습 입력 안정화)

# ZI-Tweedie 전용 — Algorithm 1(EM) 반복 수 + Algorithm 2(ζ profile likelihood) 후보 그리드
N_EM_ITERS = 10    # Generalized EM 반복 수 (Algorithm 1)
# ζ는 논문처럼 추정: 각 후보로 EM→train 로그우도 비교→최대 ζ* 선택. 후보 수↑ = 더 촘촘하나 느려짐.
ZETA_GRID  = [round(z, 2) for z in np.arange(1.1, 1.85, 0.1)]   # [1.1, 1.2, ..., 1.8] (8개)

# 트리 공통 고정 전처리 (04_default_compare와 동일 — strategy_common §1)
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# ZIT 내부 μ/π/φ LightGBM 3개 — 전부 LightGBM '라이브러리 기본값'으로 명시 (04의 트리 기본값 철학과 동일)
#   n_estimators=100, learning_rate=0.1, num_leaves=31, max_depth=-1(무제한),
#   min_child_samples=20, subsample=1.0, colsample_bytree=1.0, reg_alpha=0, reg_lambda=0
LGBM_DEFAULTS = dict(
    # μ (Tweedie mean) — 9개
    mu_n_estimators=100, mu_learning_rate=0.1, mu_num_leaves=31, mu_max_depth=-1,
    mu_min_child_samples=20, mu_subsample=1.0, mu_colsample_bytree=1.0,
    mu_reg_alpha=0.0, mu_reg_lambda=0.0,
    # π (zero 확률) — 5개
    pi_n_estimators=100, pi_learning_rate=0.1, pi_num_leaves=31, pi_max_depth=-1,
    pi_min_child_samples=20,
    # φ (dispersion) — 5개
    phi_n_estimators=100, phi_learning_rate=0.1, phi_num_leaves=31, phi_max_depth=-1,
    phi_min_child_samples=20,
)

OUT_DIR = os.path.join(OUTPUT_DIR, '0_baseline', 'default_compare')
os.makedirs(OUT_DIR, exist_ok=True)

print('N_FOLDS:', N_FOLDS, '| N_JOBS:', N_JOBS, '| N_EM_ITERS:', N_EM_ITERS)
print('ZETA_GRID:', ZETA_GRID)
print('OUT_DIR:', OUT_DIR)

N_FOLDS: 5 | N_JOBS: -1 | N_EM_ITERS: 10
ZETA_GRID: [np.float64(1.1), np.float64(1.2), np.float64(1.3), np.float64(1.4), np.float64(1.5), np.float64(1.6), np.float64(1.7), np.float64(1.8)]
OUT_DIR: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\0_baseline\default_compare


## 3. 데이터 로드 + 고정 전처리 (1회)

`preprocess.run`(cleaning + spatial imputation + outlier) → meta features. 04와 동일한 전처리 결과를 쓴다.
ZIT는 die-level로 학습하므로, unit health를 4 die에 broadcast한 `y_train_die`를 target으로 만든다.

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

# train y 극단값(1.0 1건)만 두 번째로 큰 값으로 clip
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = int((y_raw >= 1.0).sum())
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 -> {second_max:.6f} clip, {n_clipped}개 샘플')

# 고정 전처리 1회 (Stage0 제외 → cleaning → spatial impute → outlier none)
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# 메타피처 (트리: position raw 정수 + die_x/die_y 연속형) — 04와 동일
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

# die-level feature 행렬 (전처리로 결측 채워졌으므로 NaN 없음)
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val   = xs_val[feat_cols_clean].values.astype(np.float64)
X_test  = xs_test[feat_cols_clean].values.astype(np.float64)

# die→unit 매핑 key
uid_train_die = xs_train[KEY_COL].values
uid_val_die   = xs_val[KEY_COL].values
uid_test_die  = xs_test[KEY_COL].values

# unit-level 정답 (index=ufs_serial)
y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

# die-level 정답: 각 die는 자기 unit의 health를 target으로 (full-y broadcast)
y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)
assert not np.isnan(y_train_die).any(), 'y_train_die NaN — train die의 unit이 y에 없음'

print(f'[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'  unit train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')
print(f'  NaN in X_train: {int(np.isnan(X_train).sum())}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 -> 0.097417 clip, 1개 샘플
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개
    컬럼: 1031 → 926 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=30%
  제거: 5개, 잔여: 921개
    컬럼: 926 → 921 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 894개
    컬럼: 921 → 894 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 330개, 잔여: 564개
    컬럼: 894 → 564 (330개 제거)
    DataFrame: (104748, 624)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, 

## 4. ζ 추정 — profile likelihood (Algorithm 2)

각 후보 ζ로 **full-train EM**을 적합하고 train ZI-Tweedie 로그우도를 비교해 최대 ζ*를 고른다.
ζ는 LightGBM HP가 아니라 논문이 추정하는 모델 파라미터이므로 '기본값 고정'이 아니라 **추정**한다.
ζ*는 train으로만 선택 → val/test는 깨끗. (OOF는 ζ가 train 전체를 본 만큼 1 dof 미세 낙관 가능.)

In [4]:
import time

print(f'=== ζ profile likelihood (Algorithm 2) — full train, {len(ZETA_GRID)} candidates ===')
ll_by_zeta = {}
t_all = time.time()
for z in ZETA_GRID:
    t0 = time.time()
    m = ZITboostEQLRegressor(
        zeta=z, n_em_iters=N_EM_ITERS,
        random_state=SEED, n_jobs=N_JOBS, verbose=-1, device='cpu', **LGBM_DEFAULTS,
    )
    m.fit(X_train, y_train_die)
    ll_by_zeta[z] = m.score_loglik(X_train, y_train_die)
    print(f'  ζ={z:.2f}: train loglik={ll_by_zeta[z]:,.1f} ({time.time()-t0:.0f}s)')

ZETA_STAR = max(ll_by_zeta, key=ll_by_zeta.get)
print(f'\n[ζ* 선택] {ZETA_STAR:.2f} (train 로그우도 최대) | profiling {time.time()-t_all:.0f}s')

=== ζ profile likelihood (Algorithm 2) — full train, 8 candidates ===
  ζ=1.10: train loglik=77,384.1 (49s)
  ζ=1.20: train loglik=77,272.3 (112s)
  ζ=1.30: train loglik=74,862.4 (112s)
  ζ=1.40: train loglik=72,329.8 (112s)
  ζ=1.50: train loglik=71,600.8 (109s)
  ζ=1.60: train loglik=69,527.9 (105s)
  ζ=1.70: train loglik=66,981.7 (104s)
  ζ=1.80: train loglik=69,204.4 (108s)

[ζ* 선택] 1.10 (train 로그우도 최대) | profiling 811s


## 5. ζ* 5-fold refit (LightGBM 기본값)

unit ID 단위 KFold(같은 unit의 4 die가 train/val에 섞이면 leakage → 반드시 unit 단위 분할).
선택된 ζ*로 die-level full-y 학습 → die 예측 (1-π)μ → val/test는 fold 평균.

In [5]:
# unit ID 기준 KFold — hpo._make_unit_folds 와 동일 방식(KFold shuffle+seed on unique units → die mask)
unique_units = y_train_unit_s.index.values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(unique_units))

n_tr, n_vl, n_te = len(X_train), len(X_val), len(X_test)
oof_die  = np.full(n_tr, np.nan)   # train fold OOF
val_die  = np.zeros(n_vl)          # val/test: fold 평균 누적
test_die = np.zeros(n_te)

print(f'=== ZITboostEQLRegressor {N_FOLDS}-fold refit (ζ*={ZETA_STAR:.2f}, LightGBM 기본값) ===')
t_all = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    t0 = time.time()
    tr_units = unique_units[tr_uidx]
    vl_units = unique_units[vl_uidx]
    tr_mask  = np.isin(uid_train_die, tr_units)   # unit mask → die mask
    vl_mask  = np.isin(uid_train_die, vl_units)

    model = ZITboostEQLRegressor(
        zeta=ZETA_STAR, n_em_iters=N_EM_ITERS,
        random_state=SEED, n_jobs=N_JOBS, verbose=-1, device='cpu', **LGBM_DEFAULTS,
    )
    model.fit(X_train[tr_mask], y_train_die[tr_mask])   # die-level full-y 학습

    oof_die[vl_mask] = model.predict(X_train[vl_mask])  # E[Y]=(1-π)μ (내부에서 음수 clip)
    val_die  += model.predict(X_val)  / N_FOLDS         # val/test는 fold 평균
    test_die += model.predict(X_test) / N_FOLDS
    print(f'  fold {fold_idx+1}/{N_FOLDS}: tr_units={len(tr_units)}, vl_units={len(vl_units)} ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die).any(), 'OOF die 미커버 — fold 누락'
print(f'[refit 완료] {time.time()-t_all:.0f}s')

=== ZITboostEQLRegressor 5-fold refit (ζ*=1.10, LightGBM 기본값) ===
  fold 1/5: tr_units=20949, vl_units=5238 (111s)
  fold 2/5: tr_units=20949, vl_units=5238 (111s)
  fold 3/5: tr_units=20950, vl_units=5237 (113s)
  fold 4/5: tr_units=20950, vl_units=5237 (114s)
  fold 5/5: tr_units=20950, vl_units=5237 (112s)
[refit 완료] 560s


## 6. die→unit mean 집계 + RMSE → results_zit.csv 저장

die 예측을 unit 평균으로 집계해 oof/val/test RMSE를 낸다. 04 `results.csv`가 있으면 함께 비교 출력(파일은 덮어쓰지 않음).

In [6]:
# die-level 예측 → unit 평균 (Series, index=ufs_serial)
def unit_mean(uid_die, die_pred):
    return pd.DataFrame({KEY_COL: uid_die, 'v': np.asarray(die_pred)}).groupby(KEY_COL, sort=False)['v'].mean()

# 예측 Series를 정답 index 순서에 맞춰 RMSE
def rmse_unit(pred_s, y_s):
    p = pred_s.loc[y_s.index]
    return float(np.sqrt(np.mean((p.values - y_s.values) ** 2)))

oof_rmse  = rmse_unit(unit_mean(uid_train_die, oof_die),  y_train_unit_s)
val_rmse  = rmse_unit(unit_mean(uid_val_die,   val_die),  y_val_unit_s)
test_rmse = rmse_unit(unit_mean(uid_test_die,  test_die), y_test_unit_s)

# 04 results.csv 포맷에 맞춤 (mode/clf/reg/oof_rmse/val_rmse/test_rmse). ζ*는 reg 라벨에 기록.
result = pd.DataFrame([{
    'mode': 'zit', 'clf': '-', 'reg': f'zitboost_eql_z{ZETA_STAR:.2f}',
    'oof_rmse': oof_rmse, 'val_rmse': val_rmse, 'test_rmse': test_rmse,
}])

out_path = os.path.join(OUT_DIR, 'results_zit.csv')
result.to_csv(out_path, index=False)
print(f'저장: {out_path}  (1행, ζ*={ZETA_STAR:.2f})')
print(result.to_string(index=False, float_format='%.6f'))

# 04 결과가 있으면 함께 비교 출력 (results.csv는 덮어쓰지 않음 — display 전용)
ref_path = os.path.join(OUT_DIR, 'results.csv')
if os.path.exists(ref_path):
    ref = pd.read_csv(ref_path)
    combined = pd.concat([ref, result], ignore_index=True).sort_values('val_rmse').reset_index(drop=True)
    print('\n=== 04(reg/two_stage) + 05(zit) 통합 비교 (val 오름차순, head 12) ===')
    print(combined.head(12).to_string(index=False, float_format='%.6f'))

저장: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\0_baseline\default_compare\results_zit.csv  (1행, ζ*=1.10)
mode clf                reg  oof_rmse  val_rmse  test_rmse
 zit   - zitboost_eql_z1.10  0.005515  0.005722   0.008424

=== 04(reg/two_stage) + 05(zit) 통합 비교 (val 오름차순, head 12) ===
     mode      clf                reg  oof_rmse  val_rmse  test_rmse
two_stage     lgbm               lgbm  0.005523  0.005718   0.008418
two_stage      xgb               lgbm  0.005539  0.005719   0.008413
two_stage     lgbm           catboost  0.005526  0.005721   0.008419
two_stage catboost               lgbm  0.005524  0.005721   0.008419
two_stage     lgbm                xgb  0.005543  0.005722   0.008421
two_stage      xgb           catboost  0.005541  0.005722   0.008413
      zit        - zitboost_eql_z1.10  0.005515  0.005722   0.008424
two_stage      xgb                xgb  0.005559  0.005722   0.008416
two_stage catboost           catboost  0.005527  0.005723   0.008419
two_stage     lgbm     